# Stage 05 — Rigorous Evaluation Framework
## المرحلة 05 — إطار تقييم صارم (للماجستير)

يضيف هذا الدفتر صرامة بحثية فوق نتائج المرحلة 04:
1. **الدلالة الإحصائية**: فترات ثقة (bootstrap) واختبار McNemar المزدوج لمقارنة النماذج.
2. **LLM-as-judge للـ Faithfulness** (يُضاف لاحقاً): تقييم استناد الإجابة للسياق المسترجع.

> الهدف: تجنّب الادّعاءات غير المدعومة، وتقديم نتائج قابلة للدفاع أمام اللجنة.

In [1]:
# Stage 05.1 - Setup and load Stage 04 generation results
from pathlib import Path
import pandas as pd, numpy as np
from scipy import stats
import json

PROJECT_DIR = Path('saudi_labor_law_voice_agent_project')
STAGE04_DIR = PROJECT_DIR / '09_stage04_llm_generation_results'
STAGE05_DIR = PROJECT_DIR / '10_stage05_rigorous_evaluation'
STAGE05_DIR.mkdir(parents=True, exist_ok=True)

detail_path = STAGE04_DIR / 'generation_evaluation_detailed.csv'
assert detail_path.exists(), 'Run Stage 04 (notebook 444) first.'
df = pd.read_csv(detail_path, encoding='utf-8-sig')
RNG = np.random.default_rng(42)
print('Loaded generation results:', df.shape)
print('Models:', df['model_key'].unique().tolist())
print('Question types:', df['eval_type'].value_counts().to_dict())

Loaded generation results: (90, 33)
Models: ['allam_7b', 'qwen_7b']
Question types: {'legal_article_retrieval': 34, 'faq_retrieval': 28, 'out_of_scope': 24, 'small_talk': 4}


## 05.2 — الدلالة الإحصائية (Bootstrap CI + McNemar)

- **Bootstrap 95% CI**: مدى الثقة في دقة كل نموذج (10,000 إعادة معاينة).
- **McNemar exact test**: اختبار مزدوج على نفس الأسئلة — هل الفرق بين النموذجين دالّ أم صدفة؟
- قاعدة القرار: p < 0.05 ⟵ فرق دالّ إحصائياً.

In [2]:
# Stage 05.2 - Statistical significance: bootstrap CIs + paired McNemar test
def bootstrap_ci(values, n_boot=10000, alpha=0.05):
    v = np.asarray(values, dtype=float)
    if len(v) == 0:
        return (np.nan, np.nan)
    boot = np.array([RNG.choice(v, size=len(v), replace=True).mean() for _ in range(n_boot)])
    lo, hi = np.percentile(boot, [100*alpha/2, 100*(1-alpha/2)])
    return lo, hi

def paired_significance(sub, metric='answer_correct'):
    piv = sub.pivot_table(index='eval_id', columns='model_key', values=metric, aggfunc='first').dropna()
    models = list(piv.columns)
    rows = []
    for m in models:
        v = piv[m].astype(int).values
        lo, hi = bootstrap_ci(v)
        rows.append({'model': m, 'n': len(v), f'{metric}_mean': round(v.mean(),4),
                     'ci95_low': round(lo,4), 'ci95_high': round(hi,4)})
    res = pd.DataFrame(rows)
    mcnemar = None
    if len(models) == 2:
        a = piv[models[0]].astype(int); b = piv[models[1]].astype(int)
        n_ab = int(((a==1)&(b==0)).sum()); n_ba = int(((a==0)&(b==1)).sum())
        pval = stats.binomtest(min(n_ab,n_ba), n_ab+n_ba, 0.5).pvalue if (n_ab+n_ba)>0 else 1.0
        dlo, dhi = bootstrap_ci(a.values - b.values)
        mcnemar = {'model_A': models[0], 'model_B': models[1],
                   'A_better': n_ab, 'B_better': n_ba, 'mcnemar_exact_p': round(pval,4),
                   'significant_at_0.05': bool(pval < 0.05),
                   'diff_A_minus_B_mean': round((a.mean()-b.mean()),4),
                   'diff_ci95_low': round(dlo,4), 'diff_ci95_high': round(dhi,4)}
    return res, mcnemar

subsets = {'all_questions': df,
           'legal_only': df[df['eval_type']=='legal_article_retrieval'],
           'faq_only': df[df['eval_type']=='faq_retrieval']}
all_ci, all_mc = [], []
for name, sub in subsets.items():
    if len(sub)==0: continue
    res, mc = paired_significance(sub)
    res.insert(0, 'subset', name); all_ci.append(res)
    if mc: mc['subset']=name; all_mc.append(mc)
ci_table = pd.concat(all_ci, ignore_index=True)
mc_table = pd.DataFrame(all_mc)
ci_table.to_csv(STAGE05_DIR/'model_accuracy_bootstrap_ci.csv', index=False, encoding='utf-8-sig')
mc_table.to_csv(STAGE05_DIR/'model_comparison_mcnemar.csv', index=False, encoding='utf-8-sig')
print('=== Bootstrap 95% Confidence Intervals (accuracy) ===')
print(ci_table.to_string(index=False))
print()
print('=== McNemar paired significance (ALLaM vs Qwen) ===')
print(mc_table.to_string(index=False))

=== Bootstrap 95% Confidence Intervals (accuracy) ===
       subset    model  n  answer_correct_mean  ci95_low  ci95_high
all_questions allam_7b 45               0.7333    0.6000     0.8667
all_questions  qwen_7b 45               0.6889    0.5556     0.8222
   legal_only allam_7b 17               0.7647    0.5294     0.9412
   legal_only  qwen_7b 17               0.8235    0.6471     1.0000
     faq_only allam_7b 14               0.4286    0.2143     0.7143
     faq_only  qwen_7b 14               0.2143    0.0000     0.4286

=== McNemar paired significance (ALLaM vs Qwen) ===
 model_A model_B  A_better  B_better  mcnemar_exact_p  significant_at_0.05  diff_A_minus_B_mean  diff_ci95_low  diff_ci95_high        subset
allam_7b qwen_7b         3         1            0.625                False               0.0444        -0.0444          0.1333 all_questions
allam_7b qwen_7b         0         1            1.000                False              -0.0588        -0.1765          0.0000    legal

In [3]:
# Stage 05.2b - Academic interpretation (auto-generated, honest)
lines = []
for _, r in mc_table.iterrows():
    sig = 'significant' if r['significant_at_0.05'] else 'NOT significant (difference within chance range)'
    lines.append(
        f"[{r['subset']}] {r['model_A']} vs {r['model_B']}: "
        f"McNemar p={r['mcnemar_exact_p']}, accuracy diff={r['diff_A_minus_B_mean']*100:+.1f} pp "
        f"(95%CI [{r['diff_ci95_low']*100:+.1f}, {r['diff_ci95_high']*100:+.1f}]) -> {sig}."
    )
interpretation = chr(10).join(lines)
print(interpretation)
(STAGE05_DIR / 'statistical_interpretation.txt').write_text(interpretation, encoding='utf-8')
any_sig = bool(mc_table['significant_at_0.05'].any())
print()
if any_sig:
    print('Conclusion: a statistically significant difference exists in some subsets.')
else:
    print('Conclusion: NO significant difference between models at this sample size. The eval set is small; it must be expanded before any superiority claim.')


[all_questions] allam_7b vs qwen_7b: McNemar p=0.625, accuracy diff=+4.4 pp (95%CI [-4.4, +13.3]) -> NOT significant (difference within chance range).
[legal_only] allam_7b vs qwen_7b: McNemar p=1.0, accuracy diff=-5.9 pp (95%CI [-17.6, +0.0]) -> NOT significant (difference within chance range).
[faq_only] allam_7b vs qwen_7b: McNemar p=0.25, accuracy diff=+21.4 pp (95%CI [+0.0, +42.9]) -> NOT significant (difference within chance range).

Conclusion: NO significant difference between models at this sample size. The eval set is small; it must be expanded before any superiority claim.


## 05.3 — LLM-as-Judge: Faithfulness & Answer Relevance *(قيد الإضافة)*

الخطوة التالية: تقييم كل إجابة آلياً عبر نموذج حَكَم (LLM-as-judge) على:
- **Faithfulness**: هل الإجابة مستندة فعلاً للسياق المسترجع (لا هلوسة)؟
- **Answer Relevance**: هل تجيب عن السؤال فعلاً؟

مع **التحقّق من الحَكَم** مقابل عيّنة مُقيّمة بشرياً (مطلوب للماجستير).

In [4]:
# Stage 05.3a - Retrieval for judging (structural + e5_large + reranker = best config)
from sentence_transformers import SentenceTransformer, CrossEncoder
import chromadb, torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

_emb = SentenceTransformer(r"C:/models/multilingual-e5-large", device=DEVICE)
_rer = CrossEncoder(r"C:/models/bge-reranker-v2-m3", device=DEVICE)
_chroma = chromadb.PersistentClient(path=str(PROJECT_DIR / "06_chroma_db"))
_col = _chroma.get_collection("rag_structural_e5_large")

def retrieve_context(question, top_k=3, candidates=30):
    qv = _emb.encode("query: " + str(question), normalize_embeddings=True).tolist()
    res = _col.query(query_embeddings=[qv], n_results=candidates)
    docs = res["documents"][0]
    if not docs:
        return []
    scores = _rer.predict([[str(question), d] for d in docs])
    order = np.argsort(scores)[::-1][:top_k]
    return [docs[i] for i in order]

print("Retrieval for judging ready (collection rag_structural_e5_large).")


C:\Users\PC\anaconda3\envs\datacolection\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 10859.09it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 15435.25it/s]

Retrieval for judging ready (collection rag_structural_e5_large).


In [5]:
# Stage 05.3b - Load LLM judge (Qwen2.5-7B, 4-bit) and define faithfulness judge
import re
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

JUDGE_MODEL_PATH = r"C:/models/Qwen2.5-7B-Instruct"
JUDGE_MODEL_NAME = "Qwen2.5-7B-Instruct"   # NOTE: independent-judge limitation documented below

_qc = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
_jt = AutoTokenizer.from_pretrained(JUDGE_MODEL_PATH)
_jm = AutoModelForCausalLM.from_pretrained(JUDGE_MODEL_PATH, device_map="auto", quantization_config=_qc)
_jm.eval()

_JUDGE_SYS = "You are a strict evaluation judge for an Arabic legal RAG system. Judge ONLY from the provided context."

def judge_answer(question, context_chunks, answer):
    ctx_text = "\n---\n".join(context_chunks) if context_chunks else "(لا يوجد سياق)"
    user = (
        "قيّم الإجابة بناءً على السياق المسترجع فقط.\n\n"
        f"السؤال: {question}\n\n"
        f"السياق المسترجع:\n{ctx_text}\n\n"
        f"الإجابة المُقدّمة:\n{answer}\n\n"
        "أعطِ تقييمك بصيغة JSON فقط:\n"
        '{"faithfulness": <1-5>, "answer_relevance": <1-5>, "reason": "<سبب موجز>"}\n'
        "- faithfulness: هل كل ما في الإجابة مدعوم بالسياق؟ (5=كله مدعوم، 1=هلوسة)\n"
        "- answer_relevance: هل تجيب عن السؤال فعلاً؟ (5=تماماً، 1=لا)"
    )
    msgs = [{"role": "system", "content": _JUDGE_SYS}, {"role": "user", "content": user}]
    text = _jt.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = _jt(text, return_tensors="pt", truncation=True, max_length=4000).to(_jm.device)
    with torch.no_grad():
        out = _jm.generate(**inp, max_new_tokens=160, do_sample=False, pad_token_id=_jt.eos_token_id)
    gen = _jt.decode(out[0][inp["input_ids"].shape[-1]:], skip_special_tokens=True)
    m = re.search(r"\{.*\}", gen, re.DOTALL)
    if m:
        try:
            d = json.loads(m.group(0))
            d["faithfulness"] = float(d.get("faithfulness", np.nan))
            d["answer_relevance"] = float(d.get("answer_relevance", np.nan))
            return d
        except Exception:
            pass
    return {"faithfulness": np.nan, "answer_relevance": np.nan, "reason": gen[:120]}

print("LLM judge ready:", JUDGE_MODEL_NAME)


W0621 10:47:49.799000 10308 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/339 [00:00<01:51,  3.02it/s]

Loading weights:   1%|          | 2/339 [00:00<01:47,  3.13it/s]

C:\Users\PC\anaconda3\envs\datacolection\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights:   1%|▏         | 5/339 [00:00<00:39,  8.38it/s]

...

Loading weights:  91%|█████████▏| 310/339 [00:04<00:00, 80.15it/s]

Loading weights:  94%|█████████▍| 319/339 [00:04<00:00, 75.18it/s]

Loading weights:  97%|█████████▋| 329/339 [00:04<00:00, 79.18it/s]

Loading weights: 100%|██████████| 339/339 [00:04<00:00, 71.16it/s]

LLM judge ready: Qwen2.5-7B-Instruct


In [6]:
# Stage 05.3c - Judge faithfulness for every answered in-scope question
JUDGE_TYPES = ["legal_article_retrieval", "faq_retrieval"]
to_judge = df[(df["answered"] == 1) & (df["eval_type"].isin(JUDGE_TYPES))].copy().reset_index(drop=True)
print(f"Judging {len(to_judge)} answered in-scope answers ...")

records = []
for i, row in to_judge.iterrows():
    q = str(row["question"]); ans = str(row["answer"])
    ctx = retrieve_context(q, top_k=3)
    verdict = judge_answer(q, ctx, ans)
    records.append({
        "eval_id": row.get("eval_id"), "model_key": row["model_key"],
        "model_name": row["model_name"], "eval_type": row["eval_type"],
        "faithfulness": verdict.get("faithfulness"),
        "answer_relevance": verdict.get("answer_relevance"),
        "judge_reason": verdict.get("reason", ""),
    })
    if (i + 1) % 10 == 0:
        print(f"  judged {i+1}/{len(to_judge)}")

judge_df = pd.DataFrame(records)
judge_df.to_csv(STAGE05_DIR / "llm_judge_faithfulness_detailed.csv", index=False, encoding="utf-8-sig")
print("Saved per-answer judge scores:", judge_df.shape)
judge_df[["model_name", "eval_type", "faithfulness", "answer_relevance"]].head(10)


Judging 41 answered in-scope answers ...


C:\Users\PC\anaconda3\envs\datacolection\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


  judged 10/41


  judged 20/41


  judged 30/41


  judged 40/41


Saved per-answer judge scores: (41, 7)


,model_name,eval_type,faithfulness,answer_relevance
0,ALLaM-7B-Instruct-preview,faq_retrieval,5.0,5.0
1,ALLaM-7B-Instruct-preview,faq_retrieval,5.0,5.0
2,ALLaM-7B-Instruct-preview,faq_retrieval,3.0,2.0
3,ALLaM-7B-Instruct-preview,faq_retrieval,5.0,5.0
4,ALLaM-7B-Instruct-preview,faq_retrieval,3.0,4.0
5,ALLaM-7B-Instruct-preview,faq_retrieval,5.0,5.0
6,ALLaM-7B-Instruct-preview,faq_retrieval,3.0,1.0
7,ALLaM-7B-Instruct-preview,legal_article_retrieval,5.0,5.0
8,ALLaM-7B-Instruct-preview,legal_article_retrieval,3.0,2.0
9,ALLaM-7B-Instruct-preview,legal_article_retrieval,5.0,5.0


In [7]:
# Stage 05.3d - Aggregate judge scores by model (with bootstrap CIs)
def scale01(x):
    return (np.asarray(x, dtype=float) - 1.0) / 4.0   # map 1-5 -> 0-1

rows = []
for m, g in judge_df.groupby("model_name"):
    for metric in ["faithfulness", "answer_relevance"]:
        v = g[metric].dropna().values
        if len(v) == 0:
            continue
        lo, hi = bootstrap_ci(scale01(v))
        rows.append({"model": m, "metric": metric, "n": len(v),
                     "mean_1to5": round(float(np.mean(v)), 3),
                     "mean_0to1": round(float(scale01(v).mean()), 3),
                     "ci95_low": round(lo, 3), "ci95_high": round(hi, 3)})
judge_summary = pd.DataFrame(rows)
judge_summary.to_csv(STAGE05_DIR / "llm_judge_summary.csv", index=False, encoding="utf-8-sig")
print("=== LLM-as-Judge faithfulness & relevance by model ===")
print(judge_summary.to_string(index=False))
print()
print("NOTE / LIMITATION: judge =", JUDGE_MODEL_NAME,
      "- a 7B local model and one of the evaluated systems, so self-preference bias is possible.",
      "These scores are a scalable PROXY and MUST be validated against a human-rated subset (inter-annotator agreement) before final claims.")


=== LLM-as-Judge faithfulness & relevance by model ===
                    model           metric  n  mean_1to5  mean_0to1  ci95_low  ci95_high
ALLaM-7B-Instruct-preview     faithfulness 22      4.364      0.841     0.750      0.932
ALLaM-7B-Instruct-preview answer_relevance 22      4.318      0.830     0.682      0.955
      Qwen2.5-7B-Instruct     faithfulness 19      4.579      0.895     0.789      0.974
      Qwen2.5-7B-Instruct answer_relevance 19      4.474      0.868     0.711      1.000

NOTE / LIMITATION: judge = Qwen2.5-7B-Instruct - a 7B local model and one of the evaluated systems, so self-preference bias is possible. These scores are a scalable PROXY and MUST be validated against a human-rated subset (inter-annotator agreement) before final claims.
